1. Doc2Vec를 이용해서 ratings_train 데이터를 학습
2. 학습된 모델을 저장
3. 모델을 로드하고 모델을 이용하여 ratings_test 데이터를 벡터화
    - 데이터를 token화
    - infer_vector() 함수를 이용하여 벡터화
4. LSTM 모델을 이용해서 학습, 검증

In [22]:
import pandas as pd
from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

In [23]:
#모델 학습 저장
df = pd.read_csv("./ratings_train.txt", sep='\t')
df=df[:100]
komoran=Komoran()
def tokenize(text):
    allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']
    result = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(word)
    return result

In [ ]:
#토큰화 함수는 Doc2Vec에서 사용한 토큰화 함수와 같은 함수를 이용

tokenized_sentence = [ tokenize(text) for text in df['document'] ]
tokenized_sentence

In [37]:
tagged_data=[
    TaggedDocument(words=doc, tags=[str(i)]) for i, doc in enumerate(tokenized_sentence)
]

In [38]:
d2v=Doc2Vec(
    tagged_data, vector_size=64, window=5, min_count=1, workers=2, epochs=20
)
#학습된 모델 저장
d2v.save('my_model.model')

In [39]:
#학습된 모델을 로드
loaded_doc2vec=Doc2Vec.load('my_doc2vec.model')

In [40]:
#평가용 데이터셋 test 데이터 다운로드
df=pd.read_csv('./ratings_test.txt', sep='\t')
df.drop_duplicates('document', inplace=True)

In [41]:
df=df[:5000]

In [44]:
tokenized_sentences = [ tokenize(text) for text in df['document'] ]

In [ ]:
X_vector=[]
for sentence in tokenized_sentences:
    vec=loaded_doc2vec.infer_vector(sentence)
    X_vector.append(vec)    #vec 가 맞나? sentence는 아닌가..?

X=np.array(X_vector)
y=df['label'].values

In [47]:
# Dataset, collate_fn, DataLoader 생성 
class LSTMDataset(Dataset):
    def __init__(self, vectors, labels):
        self.labels = labels
        self.data = vectors
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

In [48]:
dataset = LSTMDataset(X, y)
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [49]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle = True)

In [62]:
class LSTMCLF(nn.Module):
    def __init__(self, input_dim, hidden_size, num_classes, dropout = 0.5, head_type = 'last'):
        super().__init__()

        self.head_type = head_type
        #input_dim : 벡터화 데이터 차원의 수
        
        self.lstm = nn.LSTM(input_dim, hidden_size, batch_first=True)
        # 과적합 방지용 dropout
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        #차원을 확장
        x=x.unsqueeze(1)    #[batch_size, 1, input_dim]
        # LSTM 결과 값 A, (B,C)
        lstm_out, (hidden, cell) = self.lstm(x)
        if self.head_type == 'last':
            last_hidden = hidden.squeeze(0)
        elif self.head_type == 'mean':
            # 모든 층의 값들의 평균을 구한다. 
            # lstm_out -> [batch_size, seq_len, hidden_size]
            last_hidden = torch.mean( lstm_out, dim=1 )    # [batch_size, hidden_size]
        elif self.head_type == 'max':
            last_hidden, _ = torch.max(lstm_out, dim = 1)

        dropout_hidden = self.dropout(last_hidden)

        # return self.fc(last_hidden)
        return self.fc(dropout_hidden)

In [63]:
model = LSTMCLF(input_dim=64, hidden_size=128, num_classes=2, head_type='last', dropout=0.3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [64]:
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_train = 0 
    total_train = 0 

    for inputs, labels in tqdm(train_loader, desc = f"Epoch {epoch+1} / {epochs} "):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pred = torch.argmax(output, dim=1)
        correct_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    train_acc = (correct_train / total_train) * 100
    avg_train_loss  = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)

            val_loss += loss.item()
            pred = torch.argmax(output, dim = 1)
            correct_val += (pred == labels).sum().item()
            total_val += labels.size(0)

    val_acc = (correct_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)

    if (epoch + 1) % 10 == 0:
        print(f"LSTM 에폭 결과 : Train Loss {round(avg_train_loss, 4)} Train Acc {train_acc} " )
        print(f"LSTM 에폭 결과 : Vali Loss {round(avg_val_loss, 4)} Vali Acc {val_acc}")

Epoch 10 / 50 : 100%|██████████| 63/63 [00:03<00:00, 16.03it/s]


LSTM 에폭 결과 : Train Loss 0.5302 Train Acc 72.35000000000001 
LSTM 에폭 결과 : Vali Loss 0.5403 Vali Acc 71.0


Epoch 20 / 50 : 100%|██████████| 63/63 [00:02<00:00, 26.79it/s]


LSTM 에폭 결과 : Train Loss 0.52 Train Acc 73.05 
LSTM 에폭 결과 : Vali Loss 0.5433 Vali Acc 71.0


Epoch 30 / 50 : 100%|██████████| 63/63 [00:02<00:00, 29.91it/s]


LSTM 에폭 결과 : Train Loss 0.5092 Train Acc 73.875 
LSTM 에폭 결과 : Vali Loss 0.5418 Vali Acc 71.39999999999999


Epoch 40 / 50 : 100%|██████████| 63/63 [00:01<00:00, 57.18it/s]


LSTM 에폭 결과 : Train Loss 0.4905 Train Acc 74.85000000000001 
LSTM 에폭 결과 : Vali Loss 0.5503 Vali Acc 71.8


Epoch 50 / 50 : 100%|██████████| 63/63 [00:01<00:00, 58.55it/s]


LSTM 에폭 결과 : Train Loss 0.4758 Train Acc 75.97500000000001 
LSTM 에폭 결과 : Vali Loss 0.5538 Vali Acc 72.39999999999999
